# 08. Inferencia — clasificación de un texto nuevo

En este notebook aplicamos el pipeline entrenado en el notebook `07_clasificacion_textos_ejemplo.ipynb`  para clasificar un texto nuevo procedente del dataset. Seguiremos el proceso de la parte inferior de la imagen:

<img src="imgs/007_clasificacion_textos_pipeline.png" style="width: 700px;"/>

El proceso replica exactamente las mismas transformaciones aplicadas durante el entrenamiento:

1. Cargar un texto de muestra del dataset
2. Normalizar con la misma función de preprocesamiento
3. Cargar el vectorizador BoW y transformar el texto
4. Cargar el modelo entrenado y calcular la predicción

> **Requisito previo**: ejecutar primero el notebook 07 para generar los archivos `models/vectorizer_bow_polizas_seguros.pkl` y `models/mejor_modelo_polizas_seguros.pkl`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import joblib
import spacy
from tqdm import tqdm

---

## 1. Selección aleatoria de un texto del dataset

Cargamos el CSV y extraemos una frase al azar. Conocemos la categoría real para poder verificar si la predicción es correcta.

In [2]:
df = pd.read_csv('corpus/dataset_seguros_clasificacion_textos.csv', sep='|')

muestra        = df.sample(1).iloc[0]
texto_original = muestra['texto']
categoria_real = muestra['categoria']

print(f'Texto original:  {texto_original}')
print(f'Categoría real:  {categoria_real}')

Texto original:  Buenas tardes, tras recibir un correo con el seguro de la vivienda, no tengo claro si debo esperar, enviar documentos, pagar algo o modificar datos. quiero confirmar si puedo añadir o quitar una opción sin hacer uno nuevo. aparece un expediente antiguo y no sé si está relacionado. Gracias
Categoría real:  Pólizas


---

## 2. Normalización del texto

Aplicamos la misma función `normalizar` utilizada en el entrenamiento. Este paso es crítico: si el preprocesamiento difiere del que se aplicó al corpus de entrenamiento, el vectorizador no reconocerá los términos correctamente y la predicción será errónea.

In [3]:
nlp = spacy.load('es_core_news_sm')

# Misma función de normalización del notebook 07
def normalizar(textos):
    resultado = []
    for texto in tqdm(textos):
        doc = nlp(texto.lower())
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and token.is_alpha
            and len(token.text) > 2
        ]
        resultado.append(' '.join(tokens))
    return resultado

texto_normalizado = normalizar([texto_original])[0]

print(f'Original:    {texto_original}')
print(f'\nNormalizado: {texto_normalizado}')

100%|██████████| 1/1 [00:00<00:00, 53.92it/s]

Original:    Buenas tardes, tras recibir un correo con el seguro de la vivienda, no tengo claro si debo esperar, enviar documentos, pagar algo o modificar datos. quiero confirmar si puedo añadir o quitar una opción sin hacer uno nuevo. aparece un expediente antiguo y no sé si está relacionado. Gracias

Normalizado: tarde recibir correo seguro vivienda deber esperar enviar documento pagar modificar dato querer confirmar añadir quitar opción aparecer expediente antiguo relacionado gracias


---

## 3. Vectorización con el modelo BoW

Cargamos el vectorizador entrenado y transformamos el texto con `transform` — **no** `fit_transform`. Esto aplica el vocabulario aprendido durante el entrenamiento sin modificarlo.

In [4]:
vectorizer   = joblib.load('models/vectorizer_bow_polizas_seguros.pkl')
X_inferencia = vectorizer.transform([texto_normalizado])

print(f'Dimensión del vector: {X_inferencia.shape[1]} características')
print(f'Términos activos (no cero): {X_inferencia.nnz}')

Dimensión del vector: 234 características
Términos activos (no cero): 20


---

## 4. Predicción con el modelo entrenado

Cargamos el mejor modelo guardado en el notebook 07 y calculamos la predicción sobre el vector BoW del texto normalizado.

In [5]:
modelo     = joblib.load('models/mejor_modelo_polizas_seguros.pkl')
prediccion = modelo.predict(X_inferencia)[0]

print(f'Texto:              {texto_original}')
print(f'Categoría predicha: {prediccion}')
print(f'Categoría real:     {categoria_real}')
print()
resultado = 'CORRECTO' if prediccion == categoria_real else 'INCORRECTO'
print(f'Resultado: {resultado}')

Texto:              Buenas tardes, tras recibir un correo con el seguro de la vivienda, no tengo claro si debo esperar, enviar documentos, pagar algo o modificar datos. quiero confirmar si puedo añadir o quitar una opción sin hacer uno nuevo. aparece un expediente antiguo y no sé si está relacionado. Gracias
Categoría predicha: Pólizas
Categoría real:     Pólizas

Resultado: CORRECTO
